In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import json
import re

# 1. Initialize FastAPI App
app = FastAPI(title="SkillSync AI Backend", version="1.0")

# Allow React frontend to communicate with this backend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # In production, replace with your frontend URL
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 2. Load the Dataset
try:
    with open("job_dataset.json", "r", encoding="utf-8") as file:
        job_dataset = json.load(file)
except FileNotFoundError:
    job_dataset = []
    print("Warning: job_dataset.json not found. Please ensure the file is in the same directory.")

# Extract unique roles
unique_roles = sorted(list(set(job.get("Title") for job in job_dataset)))

# 3. Pydantic Models for Request Bodies
class GapRequest(BaseModel):
    targetRole: str
    userSkills: str

class DiscoverRequest(BaseModel):
    userSkills: str

# 4. Core Algorithmic Logic
def clean_skill(skill: str) -> str:
    """Removes qualifiers to normalize skills (e.g., 'Python basics' -> 'python')"""
    skill = skill.lower()
    # Remove common qualifiers
    skill = re.sub(r'\b(basics|fundamentals|advanced|expert|intro|introductory)\b', '', skill)
    return skill.strip()

def get_aggregated_skills_for_role(role_title: str):
    """Aggregates and ranks the most requested skills for a specific role."""
    role_entries = [job for job in job_dataset if job.get("Title") == role_title]
    skill_counts = {}
    
    for entry in role_entries:
        for skill in entry.get("Skills", []):
            cleaned = clean_skill(skill)
            if cleaned:
                skill_counts[cleaned] = skill_counts.get(cleaned, 0) + 1
                
    # Sort by frequency and take top 15 to form the "Industry Standard"
    sorted_skills = sorted(skill_counts.items(), key=lambda x: x[1], reverse=True)
    return [skill[0] for skill in sorted_skills[:15]]

# 5. API Endpoints
@app.get("/")
def read_root():
    import antigravity # 🚀
    return {"message": "SkillSync AI API is running. Antigravity engaged."}

@app.get("/api/roles")
def get_roles():
    """Returns the list of all available job roles."""
    return {"roles": unique_roles}

@app.post("/api/analyze-gap")
def analyze_gap(request: GapRequest):
    """Compares user skills against the target role's industry standard."""
    if request.targetRole not in unique_roles:
        raise HTTPException(status_code=404, detail="Role not found in dataset")

    required_skills = get_aggregated_skills_for_role(request.targetRole)
    
    # Parse user skills
    user_skills = [s.strip().lower() for s in request.userSkills.split(',') if s.strip()]
    
    # Find matches and gaps
    matched_skills = [req for req in required_skills if any(req in usr or usr in req for usr in user_skills)]
    missing_skills = [req for req in required_skills if req not in matched_skills]
    
    # Calculate score
    if not required_skills:
        score = 0
    else:
        score = min(max(round((len(matched_skills) / len(required_skills)) * 100), 5), 100)

    return {
        "role": request.targetRole,
        "score": score,
        "matched": matched_skills,
        "missing": missing_skills
    }

@app.post("/api/discover-roles")
def discover_roles(request: DiscoverRequest):
    """Recommends top roles based on the user's current skills."""
    user_skills = [s.strip().lower() for s in request.userSkills.split(',') if s.strip()]
    
    if not user_skills:
        return {"recommendations": []}

    role_scores = []
    for role in unique_roles:
        required_skills = get_aggregated_skills_for_role(role)
        if not required_skills:
            continue
            
        matched = [req for req in required_skills if any(req in usr or usr in req for usr in user_skills)]
        score = round((len(matched) / len(required_skills)) * 100)
        
        role_scores.append({
            "role": role,
            "score": score,
            "matched": matched
        })
        
    # Sort by highest score and return top 3
    role_scores.sort(key=lambda x: x["score"], reverse=True)
    return {"recommendations": role_scores[:3]}

ModuleNotFoundError: No module named 'fastapi'